# Cheapest city trip with a QUBO and Qiskit

This is a **shortest-path problem**, rather than the traveling salesman problem: travel from Moscow to Nizhny Novgorod at minimum ticket cost. We do not have to visit every city or return to the start. The connections and prices below are an **illustrative toy timetable**, not actual train routes or fares. Each arrow is an allowed travel leg. The graph branches, so there are several possible trips.

The [project notes](info.md) also require passing through a city Y. Set `VIA = "Yaroslavl"` to include that rule, or `VIA = None` to remove it. The example uses five cities and seven travel legs.


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
from qiskit.primitives import StatevectorSampler
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_optimization.converters import QuadraticProgramToQubo

START, END, VIA = "Moscow", "Nizhny Novgorod", "Yaroslavl"
G = nx.DiGraph()
G.add_weighted_edges_from([
    ("Moscow", "Tver", 4), ("Moscow", "Vladimir", 2),
    ("Tver", "Yaroslavl", 3), ("Tver", "Nizhny Novgorod", 9),
    ("Vladimir", "Yaroslavl", 4), ("Vladimir", "Nizhny Novgorod", 9),
    ("Yaroslavl", "Nizhny Novgorod", 2),
], weight="cost")

# NetworkX handles the drawing, including the ticket-price labels.
pos = {"Moscow": (0, 0), "Tver": (1, 1), "Vladimir": (1, -1),
       "Yaroslavl": (2, 1), "Nizhny Novgorod": (3, 0)}
plt.figure(figsize=(11, 4))
nx.draw(G, pos, with_labels=True, node_size=2200, node_color="lightblue", arrowsize=20)
nx.draw_networkx_edge_labels(G, pos, edge_labels=nx.get_edge_attributes(G, "cost"))
plt.title("Illustrative ticket prices")
plt.show()


## Route model and QUBO

Use one binary variable $x_{uv}$ for each arrow $u\to v$: 1 means the trip uses that leg. Minimize the sum of its ticket prices. At each city, the number of selected outgoing legs minus incoming legs must be $+1$ at the start, $-1$ at the destination, and 0 elsewhere. If Y is required, exactly one selected leg must enter Y. All prices are positive, so an optimal flow contains a simple path and no extra cycle.

`QuadraticProgramToQubo` converts these equality constraints into squared penalties, yielding $Q(x)=\sum c_{uv}x_{uv}+P\sum_v(\mathrm{out}_v-\mathrm{in}_v-b_v)^2$ (plus the optional Y penalty). The penalty is larger than the sum of all prices, so any constraint violation costs more than a valid trip. This tiny example uses seven qubits, one per binary variable.


In [ ]:
edges = list(G.edges)
problem = QuadraticProgram("cheapest_city_trip")
for i in range(len(edges)):
    problem.binary_var(name=f"x{i}")
problem.minimize(linear={f"x{i}": G[u][v]["cost"] for i, (u, v) in enumerate(edges)})

# One unit of flow leaves START and arrives at END.
for city in G.nodes:
    flow = {f"x{i}": int(u == city) - int(v == city)
            for i, (u, v) in enumerate(edges)}
    balance = 1 if city == START else -1 if city == END else 0
    problem.linear_constraint(linear=flow, sense="==", rhs=balance, name=f"flow_{city}")
if VIA is not None:
    problem.linear_constraint(
        linear={f"x{i}": int(v == VIA) for i, (_, v) in enumerate(edges)},
        sense="==", rhs=1, name="visit_Y")

penalty = sum(G[u][v]["cost"] for u, v in edges) + 1
converter = QuadraticProgramToQubo(penalty=penalty)
qubo = converter.convert(problem)
print("Edge variables:", {f"x{i}": f"{u} → {v}" for i, (u, v) in enumerate(edges)})
print(f"QUBO: {qubo.get_num_binary_vars()} binary variables, "
      f"{len(qubo.linear_constraints)} remaining constraints")
print(qubo.prettyprint())


## Simulate QAOA and compare with a classical route

QAOA is a heuristic quantum optimization circuit. `StatevectorSampler` simulates its measurements locally, while COBYLA adjusts its circuit angles. The returned candidate is checked against the original route constraints. A classical NetworkX shortest-path calculation provides a reference answer. A sampled QAOA result may be infeasible or suboptimal; changing the seed, shot count, or circuit depth can change its result.


In [ ]:
qaoa = QAOA(sampler=StatevectorSampler(default_shots=8192, seed=42),
            optimizer=COBYLA(maxiter=100), reps=2)
result = MinimumEigenOptimizer(qaoa, converters=converter).solve(problem)

# A required stop splits the classical reference trip into two shortest legs.
if VIA is None:
    classical_path = nx.shortest_path(G, START, END, weight="cost")
else:
    classical_path = (nx.shortest_path(G, START, VIA, weight="cost")[:-1]
                      + nx.shortest_path(G, VIA, END, weight="cost"))
classical_cost = nx.path_weight(G, classical_path, weight="cost")
print("Classical route:", " → ".join(classical_path), "; cost:", classical_cost)

print("QAOA status:", result.status.name)
if result.status.name == "SUCCESS":
    selected_edges = [edge for bit, edge in zip(result.x, edges) if round(bit)]
    route = nx.shortest_path(nx.DiGraph(selected_edges), START, END)
    print("QAOA route:   ", " → ".join(route), "; cost:", result.fval)
    print("Matches classical minimum:", result.fval == classical_cost)
else:
    print("QAOA did not sample a feasible route. Try more shots or repetitions.")


### Notes

This is a small teaching example, not evidence of quantum advantage: QAOA is simulated on a classical computer, and NetworkX solves this graph directly. The QUBO method is useful here because it shows how ticket costs and route rules become a binary optimization problem. For larger graphs, the number of edge variables and QAOA simulation cost grow quickly.

References: [Qiskit Optimization QUBO and minimum-eigen optimizer](https://qiskit-community.github.io/qiskit-optimization/tutorials/03_minimum_eigen_optimizer.html), [NetworkX shortest paths](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.shortest_paths.generic.shortest_path.html).
